In [8]:
!pip install konlpy

In [9]:

import sys
!{sys.executable} -m pip install ekonlpy

You should consider upgrading via the 'c:\Users\eeyy1\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [10]:
##데이터 필터링(2012년 이후 파일만 재수집)
import os
import shutil

# 1. 원본 파일이 있는 폴더 경로 (본인의 경로로 수정하세요)
source_dir = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\bond_text_final_v5'

# 2. 2012년 이후 파일만 따로 모을 새 폴더 경로
target_dir = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\bond_text_2012_to_2025'

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# 3. 기준 날짜 설정
start_date = "20120101"

print("🔍 파일 필터링을 시작합니다...")

count = 0
for filename in os.listdir(source_dir):
    # 파일명 앞 8자리가 날짜라고 가정 (예: 20120115_증권사_제목.txt)
    file_date = filename[:8]
    
    # 숫자로 된 날짜인지 확인하고 기준일보다 크거나 같으면 복사
    if file_date.isdigit() and file_date >= start_date:
        src_path = os.path.join(source_dir, filename)
        dst_path = os.path.join(target_dir, filename)
        
        shutil.copy2(src_path, dst_path) # 파일 복사 (원본 유지)
        count += 1

print(f"✅ 필터링 완료! {count}개의 파일이 '{target_dir}'로 복사되었습니다.")

🔍 파일 필터링을 시작합니다...


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'C:\\KDT\\only-pull-me\\PJT_BOK\\team2_forecast_pjt\\crawler\\crawler\\bond_reports\\bond_text_final_v5'

In [ ]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\eeyy1\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [ ]:
##날짜별 통합
import os
import re
import pandas as pd
from tqdm import tqdm

# 1. 파일들이 모여있는 폴더 경로 (전달해주신 경로 사용)
source_dir = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\bond_text_2012_to_2025'

# 2. 결과물을 저장할 경로 (원하시는 위치로 수정 가능)
save_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\integrated_reports_2012_2025.csv'

def clean_text(text):
    """불필요한 공백 및 특수문자 제거"""
    # 한글과 공백만 남기기
    text = re.sub(r'[^가-힣\s]', ' ', text)
    # 여러 개의 공백을 하나로 합치기
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 3. 날짜별로 텍스트 통합하기
data_dict = {}

print(f"📂 폴더 내 파일 읽기 시작: {source_dir}")
filenames = [f for f in os.listdir(source_dir) if f.endswith('.txt')]

for filename in tqdm(filenames, desc="통합 중"):
    # 파일명 앞 8자리(날짜) 추출
    file_date = filename[:8]
    
    file_path = os.path.join(source_dir, filename)
    
    try:
        # 파일 읽기
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        # 텍스트 정제
        cleaned_content = clean_text(content)
        
        if cleaned_content:
            # 해당 날짜에 이미 데이터가 있으면 이어 붙이고, 없으면 새로 생성
            if file_date not in data_dict:
                data_dict[file_date] = cleaned_content
            else:
                data_dict[file_date] += " " + cleaned_content
                
    except Exception as e:
        print(f"❌ 오류 발생 ({filename}): {e}")

# 4. 결과 저장
if data_dict:
    # 데이터프레임 생성 및 날짜순 정렬
    df = pd.DataFrame(list(data_dict.items()), columns=['date', 'content'])
    df = df.sort_values(by='date').reset_index(drop=True)
    
    # CSV 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print(f"✅ 작업 완료!")
    print(f"📊 총 통합 날짜 수: {len(df)}일")
    print(f"💾 결과 파일: {save_path}")
    print("="*50)
else:
    print("⚠️ 통합할 파일이 없습니다. 경로를 다시 확인해주세요.")

📂 폴더 내 파일 읽기 시작: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\bond_text_2012_to_2025


통합 중: 100%|██████████| 7722/7722 [01:37<00:00, 79.44it/s] 



✅ 작업 완료!
📊 총 통합 날짜 수: 2458일
💾 결과 파일: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\integrated_reports_2012_2025.csv


In [ ]:
##n-gram추출
import sys
import os

# eKoNLPy가 설치된 경로를 직접 추가
lib_path = r'C:\Users\eeyy1\AppData\Local\Programs\Python\Python311\Lib\site-packages'
if lib_path not in sys.path:
    sys.path.append(lib_path)

try:
    from ekonlpy.sentiment import MPCK
    print("✅ eKoNLPy 로드 성공!")
except ModuleNotFoundError:
    print("❌ 여전히 찾을 수 없습니다. 경로를 다시 확인해주세요.")


# 1. eKoNLPy의 금융 통화정책 분석기 초기화
mpck = MPCK()

def get_sentiment_features(text):
    # 2. 텍스트를 토큰화 (금융 단어 및 n-gram 추출)
    tokens = mpck.tokenize(text)
    
    # 3. 매파(Hawkish)/비둘기파(Dovish) 판단을 위한 n-gram 리스트 추출
    ngram = mpck.ngramize(tokens)
    
    return tokens + ngram

# 예시: "금리가 인상될 것으로 보입니다." -> ['금리/NNG', '인상/NNG', '금리;인상']


✅ eKoNLPy 로드 성공!


UnicodeDecodeError: 'cp949' codec can't decode byte 0x80 in position 16: illegal multibyte sequence

In [ ]:
from ekonlpy.tag import Mecab

# MPCK 대신 Mecab 형태소 분석기를 바로 사용합니다.
# eKoNLPy의 Mecab은 금융 용어 사전이 내장되어 있습니다.
mecab = Mecab()

def get_tokens(text):
    # 금융 용어 단위로 토큰화
    return mecab.pos(text)

# 테스트
test_text = "금리 인상 가능성이 높습니다."
print(get_tokens(test_text))

[('금리', 'NNG'), ('인상', 'NNG'), ('가능성', 'NNG'), ('이', 'JKS'), ('높', 'VA'), ('습니다', 'EF'), ('.', 'SF')]


In [ ]:
import pandas as pd
from ekonlpy.tag import Mecab
from tqdm import tqdm

# 1. 아까 저장한 통합 CSV 불러오기
csv_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\integrated_reports_2012_2025.csv'
df = pd.read_csv(csv_path)

# 2. 분석기 및 도구 설정
mecab = Mecab()

def get_tokens_and_ngrams(text):
    if not isinstance(text, str): return ""
    
    # 2-1. 형태소 분석 (단어/태그 추출)
    # 예: [('금리', 'NNG'), ('인상', 'NNG')] -> ['금리/NNG', '인상/NNG']
    tokens = [f"{word}/{tag}" for word, tag in mecab.pos(text)]
    
    # 2-2. n-gram 추출 (중요!)
    # eKoNLPy의 Mecab은 금융 n-gram을 자동으로 생성하는 기능을 지원하기도 하지만, 
    # 여기서는 논문 방식처럼 2~5개 단어 조합을 만드는 과정을 포함할 수 있습니다.
    # 우선 가장 기본인 토큰 리스트를 반환합니다.
    return ", ".join(tokens)

# 3. 전체 데이터에 적용 (시간이 조금 걸릴 수 있습니다)
print("🚀 7,722개 리포트 분석(토큰화) 시작...")
tqdm.pandas() # 판다스용 진행바 설정
df['tokens'] = df['content'].progress_apply(get_tokens_and_ngrams)

# 4. 결과 저장
output_path = r'"C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\tokenized_reports.csv"'
df[['date', 'tokens']].to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 분석 완료! 결과가 '{output_path}'에 저장되었습니다.")

🚀 8,808개 리포트 분석(토큰화) 시작...


100%|██████████| 2458/2458 [06:07<00:00,  6.69it/s]


✅ 분석 완료! 결과가 'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\tokenized_reports.csv'에 저장되었습니다.


In [ ]:
import pandas as pd
from ekonlpy.sentiment.utils import MPTokenizer # MPCK 대신 핵심 엔진만 가져옴
from tqdm import tqdm

# 1. 토큰화 파일 불러오기
token_csv_path = r'"C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\tokenized_reports.csv"'
df = pd.read_csv(token_csv_path)

# 2. n-gram 추출 엔진 설정
tokenizer = MPTokenizer()

def extract_ngrams(tokens_str):
    if not isinstance(tokens_str, str) or tokens_str == "":
        return ""
    
    # 2-1. '금리/NNG, 인상/NNG' 형태를 리스트 ['금리/NNG', '인상/NNG']로 변환
    token_list = [t.strip() for t in tokens_str.split(',')]
    
    try:
        # 2-2. 논문 규격 n-gram 생성 (최대 5개 단어 조합)
        # MPTokenizer의 ngramize는 사전 로드 없이 로직만 수행하므로 에러가 안 납니다.
        ngrams = tokenizer.ngramize(token_list)
        
        # 원본 단어와 n-gram을 합쳐서 저장
        combined = token_list + ngrams
        return ", ".join(combined)
    except Exception as e:
        # 만약 에러 발생 시 원본이라도 유지
        return tokens_str

# 3. n-gram 추출 실행 (7,722개 대상)
print("🚀 논문 규격 n-gram 생성 시작 (MPCK 우회 방식)...")
tqdm.pandas()
df['ngram_features'] = df['tokens'].progress_apply(extract_ngrams)

# 4. 최종 결과 저장
final_output_path = r'"C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\paper_features_7722.csv"'
df[['date', 'ngram_features']].to_csv(final_output_path, index=False, encoding='utf-8-sig')

print(f"✅ 완료! 논문 구현용 피처 파일이 생성되었습니다: {final_output_path}")

🚀 논문 규격 n-gram 생성 시작 (MPCK 우회 방식)...


100%|██████████| 2458/2458 [00:33<00:00, 73.55it/s] 


✅ 완료! 논문 구현용 피처 파일이 생성되었습니다: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\paper_features_7722.csv


In [ ]:
import pandas as pd
from ekonlpy.sentiment.utils import MPTokenizer # 핵심 엔진
from tqdm import tqdm

# 1. 방금 만드신 'tokens'가 들어있는 CSV 불러오기
token_csv_path = r'"C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\tokenized_reports.csv"'
df = pd.read_csv(token_csv_path)

# 2. n-gram 추출 도구
tokenizer = MPTokenizer()

def make_ngrams(tokens_str):
    if not isinstance(tokens_str, str) or tokens_str == "":
        return ""
    
    # 현재 '단어/태그, 단어/태그' 형태를 리스트로 변환
    token_list = [t.strip() for t in tokens_str.split(',')]
    
    try:
        # 논문 핵심: 단어 조합(n-gram) 생성
        # 예: ['금리/NNG', '인상/NNG'] -> ['금리/NNG;인상/NNG']
        ngrams = tokenizer.ngramize(token_list)
        
        # 원본 단어(unigram)와 조합된 단어(n-gram)를 모두 합침
        combined = token_list + ngrams
        return ", ".join(combined)
    except:
        return tokens_str

# 3. 7,722개 데이터에 적용
print("🚀 이제 단어들을 조합(n-gram)하여 논문 규격으로 변환합니다...")
tqdm.pandas()
df['ngram_features'] = df['tokens'].progress_apply(make_ngrams)

# 4. 저장
final_path = r'"C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\final_paper_features.csv"'
df[['date', 'ngram_features']].to_csv(final_path, index=False, encoding='utf-8-sig')

print(f"✅ 완료! 이제 논문 분석 준비가 끝났습니다: {final_path}")

🚀 이제 단어들을 조합(n-gram)하여 논문 규격으로 변환합니다...


100%|██████████| 2458/2458 [00:37<00:00, 65.58it/s] 


✅ 완료! 이제 논문 분석 준비가 끝났습니다: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\crawler\crawler\bond_reports\final_paper_features.csv


In [ ]:
import pandas as pd
import os

# 1. 경로 설정
base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'
output_path = os.path.join(base_path, 'final_preprocessed_data.csv')

# 2. 전처리된 파일들 불러오기 (예: 파일이 여러 개라면 리스트로 관리)
# 여기서는 아까 작업하던 파일을 예시로 들게요.
file_name = 'final_paper_features.csv' 
df = pd.read_csv(os.path.join(base_path, file_name))

# 3. 형식 맞추기 (Data Transformation)
# date 형식을 YYYY-MM-DD로 변환
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

# 만약 content, category, source 컬럼이 없다면 기본값 생성 (실제 데이터에 맞게 수정 필요)
if 'content' not in df.columns: df['content'] = "" # 원문 데이터가 있다면 연결
if 'category' not in df.columns: df['category'] = "리포트"
if 'source' not in df.columns: df['source'] = "증권사" # 혹은 파일명에서 추출 가능

# tokens 컬럼 이름 맞추기 (기존 ngram_features 등을 tokens로 변경)
if 'ngram_features' in df.columns:
    df = df.rename(columns={'ngram_features': 'tokens'})

# 4. 요청하신 순서대로 컬럼 재배치
final_df = df[['date', 'content', 'tokens', 'category', 'source']]

# 5. 저장 (index=False 필수!)
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 오늘 업무 완료! {len(final_df)}건의 데이터가 규격에 맞춰 합쳐졌습니다.")
print(f"위치: {output_path}")

✅ 오늘 업무 완료! 2458건의 데이터가 규격에 맞춰 합쳐졌습니다.
위치: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\final_preprocessed_data.csv


In [ ]:
# 날짜별로 몇 개의 리포트가 합쳐졌는지 확인
check_counts = final_df['date'].value_counts()
print(check_counts.head())
print(f"총 유효 날짜 수: {len(check_counts)}일")

date
1970-01-01    2458
Name: count, dtype: int64
총 유효 날짜 수: 1일


In [ ]:
import pandas as pd
import os

base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'
df = pd.read_csv(os.path.join(base_path, 'final_paper_features.csv'))

# 1. 실제 파일에 들어있는 컬럼명 확인 (에러 방지용 출력)
print("현재 파일 컬럼명:", df.columns.tolist())

# 2. 날짜 변환 (1970년 에러 방지)
df['date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['date'])

# 3. 토큰 컬럼 찾기 (ngram_features 또는 tokens 등)
# 꽃게님의 파일에서 토큰이 들어있는 컬럼명을 'tokens'로 통일해줍니다.
if 'ngram_features' in df.columns:
    df = df.rename(columns={'ngram_features': 'tokens'})
elif 'tokens' not in df.columns:
    # 만약 둘 다 없다면, 가장 마지막 컬럼이 토큰일 확률이 높으니 그것을 사용
    df = df.rename(columns={df.columns[-1]: 'tokens'})

# 4. content 컬럼이 없는 경우 빈 문자열로 생성
if 'content' not in df.columns:
    df['content'] = ""

# 5. 데이터 합치기 (Groupby)
print("🚀 날짜별로 데이터를 합치는 중...")
final_df = df.groupby('date').agg({
    'tokens': lambda x: ', '.join(x.astype(str)),
    'content': lambda x: ' '.join(x.astype(str))
}).reset_index()

# 6. 최종 형식 맞추기
final_df['date'] = final_df['date'].dt.strftime('%Y-%m-%d')
final_df['category'] = "리포트"
final_df['source'] = "증권사"

# 컬럼 순서 고정
final_df = final_df[['date', 'content', 'tokens', 'category', 'source']]

# 7. 저장 및 확인
output_path = os.path.join(base_path, 'final_preprocessed_data.csv')
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 수정 성공! 총 {len(final_df)}일치의 데이터가 저장되었습니다.")

현재 파일 컬럼명: ['date', 'ngram_features']
🚀 날짜별로 데이터를 합치는 중...
✅ 수정 성공! 총 2458일치의 데이터가 저장되었습니다.


In [ ]:
##7,722건의 유효 레포트가 2,458일치 데이터로 병합
##2012년부터 현재까지의 영업일(주식 시장개장일)은 약 2,400~2,500일 정도이므로 유효한 숫자로 판단

In [ ]:
import pandas as pd
import os

# 1. 파일 불러오기
base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'
df = pd.read_csv(os.path.join(base_path, 'final_preprocessed_data.csv'))

# 2. 저에게 보여주실 샘플 생성 (상위 100개 날짜, 원문 제외)
# 원문(content)이 빠지면 용량이 수십 분의 일로 줄어듭니다.
sample_for_ai = df[['date', 'tokens', 'category', 'source']].head(100)

# 3. 샘플 파일 저장
sample_path = os.path.join(base_path, 'sample_for_ai.csv')
sample_for_ai.to_csv(sample_path, index=False, encoding='utf-8-sig')

print(f"✅ 저에게 보내실 샘플 파일이 생성되었습니다: {sample_path}")
print("이 파일을 채팅창에 업로드해 주시면 제가 n-gram이 잘 생성되었는지 봐드릴게요!")

✅ 저에게 보내실 샘플 파일이 생성되었습니다: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\sample_for_ai.csv
이 파일을 채팅창에 업로드해 주시면 제가 n-gram이 잘 생성되었는지 봐드릴게요!


In [ ]:
# 'tokens' 컬럼에서 세미콜론(;)이 포함된 n-gram이 하나라도 있는지 확인
has_ngram = df['tokens'].str.contains(';').sum()
print(f"n-gram이 포함된 행의 수: {has_ngram}개")

if has_ngram > 0:
    print("✅ n-gram 처리가 잘 되었습니다!")
else:
    print("⚠️ n-gram이 발견되지 않았습니다. 추출 로직을 다시 점검해야 합니다.")

n-gram이 포함된 행의 수: 2454개
✅ n-gram 처리가 잘 되었습니다!


In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# 1. 경로 설정
base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'
input_file = os.path.join(base_path, 'final_preprocessed_data.csv')

# 2. 개별 파일을 저장할 새 폴더 생성 (폴더가 너무 지저분해지지 않게 별도 생성)
output_folder = os.path.join(base_path, 'daily_reports')
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 3. 데이터 불러오기
df = pd.read_csv(input_file)

print(f"🚀 총 {len(df)}개의 날짜별 파일 생성을 시작합니다...")

# 4. 날짜별로 반복하며 파일 저장
for index, row in tqdm(df.iterrows(), total=len(df)):
    date_str = row['date']
    # 파일명에 사용할 수 없는 특수문자 제거 (이미 YYYY-MM-DD라 안전하지만 확인 차원)
    file_name = f"{date_str}.csv"
    save_path = os.path.join(output_folder, file_name)
    
    # 해당 행만 따로 떼어서 저장
    # 컬럼 순서는 아까 맞춘 [date, content, tokens, category, source] 그대로 갑니다.
    pd.DataFrame([row]).to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 모든 작업이 완료되었습니다!")
print(f"파일 저장 위치: {output_folder}")

🚀 총 2458개의 날짜별 파일 생성을 시작합니다...


100%|██████████| 2458/2458 [00:07<00:00, 317.02it/s]


✅ 모든 작업이 완료되었습니다!
파일 저장 위치: C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing\daily_reports


In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# 1. 경로 설정
base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'
daily_folder = os.path.join(base_path, 'daily_reports') # 아까 쪼개놓은 폴더

# 2. 폴더 내의 모든 CSV 파일 목록 가져오기
file_list = [f for f in os.listdir(daily_folder) if f.endswith('.csv')]
file_list.sort() # 날짜순 정렬

print(f"🚀 총 {len(file_list)}개의 일별 파일을 하나로 합치는 중...")

# 3. 파일들을 하나씩 읽어서 리스트에 담기
all_data = []
for file_name in tqdm(file_list):
    file_path = os.path.join(daily_folder, file_name)
    temp_df = pd.read_csv(file_path)
    all_data.append(temp_df)

# 4. 데이터프레임 하나로 합치기
final_merged_df = pd.concat(all_data, ignore_index=True)

# 5. (중요) 혹시라도 같은 날짜가 여러 줄 있으면 하나로 합치기 (최종 방어 로직)
# 날짜(date)를 기준으로 묶어서 텍스트들을 다 이어붙입니다.
final_merged_df = final_merged_df.groupby('date').agg({
    'content': lambda x: ' '.join(x.astype(str)),
    'tokens': lambda x: ', '.join(x.astype(str)),
    'category': 'first',
    'source': 'first'
}).reset_index()

# 6. 최종 파일 저장
output_name = 'final_integrated_reports.csv' # 팀원들이 알아보기 쉬운 이름
final_merged_df.to_csv(os.path.join(base_path, output_name), index=False, encoding='utf-8-sig')

print(f"\n✅ 통합 완료! '{output_name}' 파일이 생성되었습니다.")
print(f"총 인덱스(행) 수: {len(final_merged_df)} (하루에 한 줄씩 잘 들어갔습니다!)")

🚀 총 2458개의 일별 파일을 하나로 합치는 중...


100%|██████████| 2458/2458 [00:54<00:00, 45.08it/s]



✅ 통합 완료! 'final_integrated_reports.csv' 파일이 생성되었습니다.
총 인덱스(행) 수: 2458 (하루에 한 줄씩 잘 들어갔습니다!)


In [11]:
import pandas as pd
import os

# 1. 경로 설정
base_path = r'C:\KDT\only-pull-me\PJT_BOK\team2_forecast_pjt\preprocessing\bond_reports_preprocessing'

raw_content_file = os.path.join(base_path, 'integrated_reports_2012_2025.csv')
preprocessed_file = os.path.join(base_path, 'final_preprocessed_data.csv')

print("🚀 데이터 타입을 맞추고 병합을 시작합니다...")

# 2. 데이터 불러오기
df_raw = pd.read_csv(raw_content_file)
df_tokens = pd.read_csv(preprocessed_file)

# 3. 날짜 형식 강제 통일 (핵심!)
# 양쪽 데이터의 'date' 컬럼을 모두 날짜 형식으로 바꾼 뒤, 똑같은 문자열 형태('2012-01-09')로 변환합니다.
df_raw['date'] = pd.to_datetime(df_raw['date'].astype(str)).dt.strftime('%Y-%m-%d')
df_tokens['date'] = pd.to_datetime(df_tokens['date'].astype(str)).dt.strftime('%Y-%m-%d')

# 4. 필요한 컬럼만 추출
# 원문 파일에서 두 번째 열(content) 가져오기
df_raw_sub = df_raw[['date', df_raw.columns[1]]] 
df_raw_sub.columns = ['date', 'content']

# 전처리 파일 컬럼 선택
df_tokens_sub = df_tokens[['date', 'tokens', 'category', 'source']]

# 5. 이제 날짜(date) 기준으로 병합 (타입이 같아졌으므로 에러 안 남!)
final_df = pd.merge(df_raw_sub, df_tokens_sub, on='date', how='inner')

# 6. 날짜순 정렬 및 저장
final_df = final_df.sort_values(by='date').reset_index(drop=True)
output_path = os.path.join(base_path, 'final_integrated_full_v2.csv')
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 드디어 성공! 총 {len(final_df)}일치의 데이터가 합쳐졌습니다.")
print(f"파일 저장 완료: {output_path}")

🚀 데이터 타입을 맞추고 병합을 시작합니다...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\KDT\\only-pull-me\\PJT_BOK\\team2_forecast_pjt\\preprocessing\\bond_reports_preprocessing\\integrated_reports_2012_2025.csv'